# Arbres de la taxonomie des sprites

Ce notebook lit [`taxonomies.json`](taxonomies.json) et affiche un arbre par dimension. Les dimensions décrivent des aspects indépendants d’une image : par exemple, **la vue** indique l’angle d’observation, tandis que **la pose** décrit la position du corps. Elles ne constituent pas une seule chaîne de catégories.

Les branches regroupent les possibilités ; seuls les nœuds terminaux, indiqués en bleu, sont attribués aux images. `unknown` signifie que l’information est incertaine ; `not_applicable` indique qu’elle n’a pas de sens pour l’image. Modifier le JSON puis relancer les cellules actualise les arbres.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
dataset_dir = next((p for p in (cwd, cwd / "notebook/dataset", cwd / "dataset") if (p / "taxonomies.json").is_file()), None)
if dataset_dir is None:
    raise FileNotFoundError("Ouvre le notebook depuis notebook/dataset, notebook ou la racine du projet.")
taxonomy = json.loads((dataset_dir / "taxonomies.json").read_text(encoding="utf-8"))
print(f"Taxonomie version {taxonomy['version']} · {len(taxonomy['hierarchies'])} dimensions")


## Lire les arbres

La racine nomme la dimension, les niveaux intermédiaires regroupent les concepts et les feuilles correspondent aux valeurs enregistrées. Les noms affichés sont les identifiants exacts du JSON ; les descriptions au-dessus de chaque arbre précisent quand utiliser la dimension.


In [ ]:
def draw_tree(dimension, tree):
    # Place leaves on separate rows, then center each branch above its children.
    nodes, edges = [], []
    next_row = 0
    def visit(name, children, depth):
        nonlocal next_row
        positions = []
        for child, subtree in children.items():
            child_y = visit(child, subtree, depth + 1)
            positions.append((child, child_y))
        if positions:
            y = sum(pos for _, pos in positions) / len(positions)
            edges.extend((depth, y, depth + 1, child_y) for _, child_y in positions)
        else:
            y = next_row
            next_row += 1
        nodes.append((name, depth, y, not bool(children)))
        return y
    visit(dimension, tree, 0)
    depth_max = max(depth for _, depth, _, _ in nodes)
    fig, ax = plt.subplots(figsize=(max(9, (depth_max + 1) * 3.0), max(2.6, next_row * 0.52 + 1.0)))
    ax.set_xlim(-0.55, depth_max + 0.85)
    ax.set_ylim(next_row - 0.35, -0.65)
    ax.axis("off")
    for x1, y1, x2, y2 in edges:
        mid = (x1 + x2) / 2
        ax.plot([x1, mid, mid, x2], [y1, y1, y2, y2], color="#94a3b8", lw=1.1, zorder=1)
    for name, x, y, is_leaf in nodes:
        ax.text(x, y, name, ha="center", va="center", fontsize=9,
                color="#1e3a8a" if is_leaf else "#334155",
                bbox={"boxstyle": "round,pad=0.35", "facecolor": "#dbeafe" if is_leaf else "#f1f5f9", "edgecolor": "#93c5fd" if is_leaf else "#cbd5e1"}, zorder=2)
    plt.tight_layout()
    plt.show()

for dimension, tree in taxonomy["hierarchies"].items():
    info = taxonomy["dimensions"][dimension]
    display(Markdown(f"### {info['label']} (`{dimension}`)\n\n{info['definition']}\n\n**Application :** {info['applicability']}"))
    draw_tree(dimension, tree)
